In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project src folder was not found.")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

from sklearn.model_selection import train_test_split

from src.preprocessing import prepare_dataset
from src.three_way_split import split_three_ways

from src.feature_selection import (
    correlation_screen,
    mutual_information_scores,
    MutualInformationSelector
)

from src.models_baseline_a import (
    build_logistic_regression_pipeline,
    build_knn_pipeline
)

RANDOM_STATE = 42
TARGET = "sublabel"

Path("results/tables").mkdir(parents=True, exist_ok=True)

Project Root: C:\campus\machine learning\chronic-disease-ml


In [6]:
from pathlib import Path

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Data Warehouse Multiclass.csv"

print("Dataset path:", DATA_PATH)
print("File exists:", DATA_PATH.is_file())

X, y = prepare_dataset(DATA_PATH)

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

Dataset path: C:\campus\machine learning\chronic-disease-ml\data\raw\Data Warehouse Multiclass.csv
File exists: True
=== DATASET PREPARATION ===
Original shape: (280985, 39)
Predictor shape: (280985, 24)
Target shape: (280985,)
Exact duplicates: 2284
Target classes: 8

Class distribution:
sublabel
N           106884
HY           69806
DI_HY        66473
DI           31918
HT            2595
DI_HT         1398
HT_HY         1066
DI_HT_HY       845
Name: count, dtype: int64
Dataset shape: (280985, 24)
Target shape: (280985,)

Class distribution:
sublabel
N           106884
HY           69806
DI_HY        66473
DI           31918
HT            2595
DI_HT         1398
HT_HY         1066
DI_HT_HY       845
Name: count, dtype: int64


In [7]:
X_train, X_val, X_test, y_train, y_val, y_test = (
    split_three_ways(
        X,
        y,
        random_state=RANDOM_STATE
    )
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

# Test data must remain untouched during model selection.s


=== DUPLICATE HANDLING ===
Original records: 280985
Exact duplicate records removed: 2284
Remaining records: 278701

=== 70/15/15 DATA SPLIT ===
Training: 195091 (70.00%)
Validation: 41806 (15.00%)
Testing: 41804 (15.00%)

=== CLASS DISTRIBUTION (%) ===
            Original  Training  Validation  Testing
__target__                                         
N              37.55     37.55       37.55    37.55
HY             25.04     25.04       25.04    25.04
DI_HY          23.85     23.85       23.85    23.85
DI             11.45     11.45       11.45    11.45
HT              0.93      0.93        0.93     0.93
DI_HT           0.50      0.50        0.50     0.50
HT_HY           0.38      0.38        0.38     0.38
DI_HT_HY        0.30      0.30        0.30     0.30

SUCCESS: No overlapping predictor groups.
SUCCESS: All eight classes represented.
Training: (195091, 24)
Validation: (41806, 24)
Testing: (41804, 24)

Training class distribution:
__target__
N           73254
HY          488

In [8]:

correlation_results = correlation_screen(
    X_train,
    threshold=0.90
)

correlation_results.to_csv(
    "results/tables/high_correlation_pairs.csv",
    index=False
)

print("Highly correlated feature pairs:")
display(correlation_results)


Highly correlated feature pairs:


,Feature_1,Feature_2,Absolute_Correlation


In [9]:

# Fit preprocessing using training data only.
mi_preprocessor = build_logistic_regression_pipeline(
    X_train,
    use_engineering=False,
    selected=False
).named_steps["preprocessor"]

X_train_transformed = mi_preprocessor.fit_transform(
    X_train,
    y_train
)

feature_names = mi_preprocessor.get_feature_names_out()

mi_results = mutual_information_scores(
    X_train_transformed,
    y_train,
    feature_names=feature_names
)

mi_results.to_csv(
    "results/tables/mutual_information_scores.csv",
    index=False
)

display(mi_results.head(30))


,Feature,Mutual_Information
0,numeric__HbA1c_level,0.740108
1,numeric__crp_level,0.694608
2,numeric__homocysteine_level,0.692411
3,numeric__heart_rate,0.426761
4,numeric__diastolic_bp,0.425801
5,numeric__hdl,0.425712
6,numeric__salt_intake,0.425232
7,numeric__systolic_bp,0.424603
8,numeric__alcohol_intake,0.424407
9,numeric__ldl,0.424293


In [11]:
# Get selected feature names using the selector support mask

import numpy as np
import pandas as pd

selected_features = np.asarray(feature_names)[
    mi_selector.get_support()
]

selected_feature_table = pd.DataFrame({
    "Selected_Feature": selected_features
})

selected_feature_table.to_csv(
    PROJECT_ROOT / "results/tables/selected_feature_lists.csv",
    index=False
)

print("Number of selected features:", len(selected_features))
print("\nSelected Features:")
display(selected_feature_table)

Number of selected features: 20

Selected Features:


,Selected_Feature
0,numeric__age
1,numeric__bmi
2,numeric__HbA1c_level
3,numeric__glucose
4,numeric__cholesterol
5,numeric__sleep_hours
6,numeric__triglycerides
7,numeric__crp_level
8,numeric__homocysteine_level
9,numeric__systolic_bp


In [12]:

def evaluate_model(
    model,
    X_train_data,
    y_train_data,
    X_validation,
    y_validation,
    model_name,
    configuration
):
    model.fit(
        X_train_data,
        y_train_data
    )

    predictions = model.predict(X_validation)

    metrics = {
        "Model": model_name,
        "Configuration": configuration,
        "Accuracy": accuracy_score(
            y_validation,
            predictions
        ),
        "Macro_F1": f1_score(
            y_validation,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Weighted_F1": f1_score(
            y_validation,
            predictions,
            average="weighted",
            zero_division=0
        ),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_validation,
            predictions
        )
    }

    report = classification_report(
        y_validation,
        predictions,
        output_dict=True,
        zero_division=0
    )

    class_report = pd.DataFrame(report).transpose()

    return metrics, class_report, model


In [13]:

KNN_TRAIN_LIMIT = min(10000, len(X_train))

if len(X_train) > KNN_TRAIN_LIMIT:
    X_knn_train, _, y_knn_train, _ = train_test_split(
        X_train,
        y_train,
        train_size=KNN_TRAIN_LIMIT,
        random_state=RANDOM_STATE,
        stratify=y_train
    )
else:
    X_knn_train = X_train.copy()
    y_knn_train = y_train.copy()

print("Logistic Regression training rows:", len(X_train))
print("KNN training rows:", len(X_knn_train))
print("Validation rows:", len(X_val))


Logistic Regression training rows: 195091
KNN training rows: 10000
Validation rows: 41806


In [14]:

experiments = []
class_reports = []
fitted_models = {}

configurations = [
    ("Original", False, False),
    ("Engineered", True, False),
    ("Engineered_Selected", True, True)
]

for configuration, use_engineering, selected in configurations:

    # Logistic Regression
    lr_model = build_logistic_regression_pipeline(
        X_train,
        use_engineering=use_engineering,
        selected=selected,
        k_features=K_FEATURES
    )

    lr_metrics, lr_report, lr_model = evaluate_model(
        lr_model,
        X_train,
        y_train,
        X_val,
        y_val,
        "Logistic Regression",
        configuration
    )

    experiments.append(lr_metrics)
    lr_report["Model"] = "Logistic Regression"
    lr_report["Configuration"] = configuration
    class_reports.append(lr_report)

    fitted_models[
        ("Logistic Regression", configuration)
    ] = lr_model

    # KNN
    knn_model = build_knn_pipeline(
        X_train,
        use_engineering=use_engineering,
        selected=selected,
        k_features=K_FEATURES,
        n_neighbors=7,
        weights="distance"
    )

    knn_metrics, knn_report, knn_model = evaluate_model(
        knn_model,
        X_knn_train,
        y_knn_train,
        X_val,
        y_val,
        "KNN",
        configuration
    )

    experiments.append(knn_metrics)
    knn_report["Model"] = "KNN"
    knn_report["Configuration"] = configuration
    class_reports.append(knn_report)

    fitted_models[
        ("KNN", configuration)
    ] = knn_model

print("All baseline experiments completed.")


All baseline experiments completed.


In [15]:

metrics_df = pd.DataFrame(experiments)

metrics_df = metrics_df.sort_values(
    by="Macro_F1",
    ascending=False
)

metrics_df.to_csv(
    "results/tables/baseline_validation_metrics.csv",
    index=False
)

display(metrics_df)


,Model,Configuration,Accuracy,Macro_F1,Weighted_F1,Balanced_Accuracy
2,Logistic Regression,Engineered,0.489140,0.296151,0.555105,0.475480
1,KNN,Original,0.613859,0.287209,0.601044,0.284667
3,KNN,Engineered,0.605846,0.285457,0.593990,0.281991
4,Logistic Regression,Engineered_Selected,0.468043,0.284018,0.537431,0.443879
5,KNN,Engineered_Selected,0.611395,0.280967,0.599626,0.280311
0,Logistic Regression,Original,0.463402,0.273873,0.531376,0.427119


In [16]:

class_results = pd.concat(
    class_reports,
    axis=0
)

class_results.to_csv(
    "results/tables/class_wise_validation_metrics.csv"
)

display(class_results)


,precision,recall,f1-score,support,Model,Configuration
DI,0.283328,0.170079,0.212560,4786.000000,Logistic Regression,Original
DI_HT,0.039546,0.483254,0.073109,209.000000,Logistic Regression,Original
DI_HT_HY,0.017010,0.409449,0.032663,127.000000,Logistic Regression,Original
DI_HY,0.693387,0.515294,0.591220,9971.000000,Logistic Regression,Original
HT,0.048126,0.489691,0.087638,388.000000,Logistic Regression,Original
...,...,...,...,...,...,...
HY,0.541413,0.586358,0.562990,10468.000000,KNN,Engineered_Selected
N,0.775713,0.774033,0.774872,15697.000000,KNN,Engineered_Selected
accuracy,0.611395,0.611395,0.611395,0.611395,KNN,Engineered_Selected
macro avg,0.298544,0.280311,0.280967,41806.000000,KNN,Engineered_Selected


In [17]:

best_result = metrics_df.iloc[0]

print("Best validation configuration")
print("-----------------------------")
print("Model:", best_result["Model"])
print("Configuration:", best_result["Configuration"])
print("Macro F1:", round(best_result["Macro_F1"], 4))
print("Accuracy:", round(best_result["Accuracy"], 4))
print("Weighted F1:", round(best_result["Weighted_F1"], 4))
print(
    "Balanced Accuracy:",
    round(best_result["Balanced_Accuracy"], 4)
)

print(
    "\nNote: The test set has not been used for "
    "feature selection or model selection."
)


Best validation configuration
-----------------------------
Model: Logistic Regression
Configuration: Engineered
Macro F1: 0.2962
Accuracy: 0.4891
Weighted F1: 0.5551
Balanced Accuracy: 0.4755

Note: The test set has not been used for feature selection or model selection.


In [18]:
import pandas as pd
import numpy as np
import sklearn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("All packages imported successfully!")

Pandas: 3.1.1
NumPy: 2.5.4
Scikit-learn: 1.9.1
All packages imported successfully!


In [24]:
from pathlib import Path
import shutil

# Project root directory
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Current incorrect location
SOURCE = PROJECT_ROOT / "notebooks" / "results" / "tables"

# Correct location
DESTINATION = PROJECT_ROOT / "results" / "tables"

DESTINATION.mkdir(parents=True, exist_ok=True)

files = [
    "baseline_validation_metrics.csv",
    "class_wise_validation_metrics.csv",
    "high_correlation_pairs.csv",
    "mutual_information_scores.csv",
    "selected_feature_lists.csv"
]

for filename in files:
    source_file = SOURCE / filename
    destination_file = DESTINATION / filename

    if source_file.exists():
        shutil.copy2(source_file, destination_file)
        print("COPIED:", filename)
    else:
        print("SOURCE FILE NOT FOUND:", filename)

print("\nCorrect folder:", DESTINATION)
print("\nFinal files:")

for file in sorted(DESTINATION.glob("*.csv")):
    print("FOUND:", file.name)

COPIED: baseline_validation_metrics.csv
COPIED: class_wise_validation_metrics.csv
COPIED: high_correlation_pairs.csv
COPIED: mutual_information_scores.csv
COPIED: selected_feature_lists.csv

Correct folder: c:\campus\machine learning\chronic-disease-ml\results\tables

Final files:
FOUND: baseline_validation_metrics.csv
FOUND: categorical_summary.csv
FOUND: class_counts.csv
FOUND: class_wise_validation_metrics.csv
FOUND: correlation_matrix.csv
FOUND: data_dictionary.csv
FOUND: dataset_summary.csv
FOUND: high_correlation_pairs.csv
FOUND: missing_values.csv
FOUND: mutual_information_scores.csv
FOUND: numerical_quality_audit.csv
FOUND: part2_engineered_feature_catalogue.csv
FOUND: part2_full_feature_comparison.csv
FOUND: part2_transformed_feature_catalogue.csv
FOUND: selected_feature_lists.csv
FOUND: source_class_counts.csv
FOUND: source_class_percentages.csv
FOUND: source_statistics.csv
